# E3 — verificación de la demo con pesos reales (PREPARADO, NO EJECUTADO)
Ejecutar en Colab con GPU (Entorno de ejecución → T4). Objetivo: una máscara de punta a punta con SAM 2.1 tiny real y puntos manuales.
**Salida esperada:** `OK e3_demo_smoke` y el archivo `/content/e3_demo_smoke.json` (descargarlo). Opcional: el brazo ajustado, si se sube el `best/payload` de E2.


In [ ]:
import os, subprocess, sys
from pathlib import Path

assert sys.version_info[:2] == (3, 12), 'Selecciona un runtime Python 3.12 con GPU T4'
subprocess.run(['nvidia-smi', '-L'], check=True)

os.environ['AUTO_PROMPT_EXECUTION_CONTEXT'] = 'google_colab_user_notebook'
os.environ['SAM2_BUILD_CUDA'] = '0'
repo = Path('/content/auto-prompt-sam2-e3-demo')
if not repo.exists():
    subprocess.run(['git', 'clone', '-q', '-b', '_E.V_',
                    'https://github.com/roberto-chavez/auto-prompt-sam2', str(repo)], check=True)
os.chdir(repo)
sys.path.insert(0, str(repo / 'src'))
pip = [sys.executable, '-m', 'pip', 'install', '-q']
subprocess.run([*pip, '-r', 'requirements-demo-inference.txt',
                'setuptools==80.9.0', 'wheel==0.45.1'], check=True)
subprocess.run([*pip, '--no-build-isolation', '--no-deps',
                'git+https://github.com/facebookresearch/sam2.git@2b90b9f5ceec907a1c18123530e92e794ad901a4'], check=True)
import torch
from sam2.build_sam import build_sam2
assert torch.cuda.is_available(), 'Activa una GPU en Colab'
print(torch.cuda.get_device_name(0))


Sube un chip Sen1Floods11 de **validación** (p. ej. `Ghana_1033830_S2Hand.tif`, SHA-256 `aebdf646…3c69`) a `/content/chip.tif`, desde tu Drive o con el panel de archivos. No lo subas a Git.

In [ ]:
import hashlib, json, shutil
from pathlib import Path

chip = Path('/content/chip.tif')
if not chip.is_file():
    from google.colab import drive
    drive.mount('/content/drive')
    project = Path('/content/drive/MyDrive/auto-prompt-sam2')
    relative = Path('raw/v1.1/data/flood_events/HandLabeled/S2Hand/Ghana_1033830_S2Hand.tif')
    candidates = (project / relative, project / 'auto-prompt-sam2' / relative)
    source = next((path for path in candidates if path.is_file()), None)
    if source is None:
        raise FileNotFoundError('Sube Ghana_1033830_S2Hand.tif a /content/chip.tif')
    shutil.copyfile(source, chip)
print('sha256 chip', hashlib.sha256(chip.read_bytes()).hexdigest())


In [ ]:
from auto_prompt_sam2.demo_config import load_demo_config
from auto_prompt_sam2.demo_backend import DemoBackend
from auto_prompt_sam2.demo_images import load_image, S2_LAYOUT
cfg=load_demo_config(environ={**os.environ,'DEMO_MODE':'inference','DEMO_ALLOW_DOWNLOAD':'1'})
be=DemoBackend(cfg)
img=load_image('/content/chip.tif',S2_LAYOUT,be.params)
print(img.rgb.shape, 'válidos', int(img.valid.sum()))
# Puntos manuales (x,y,etiqueta): ajustar tras mirar img.preview(); 1=agua, 0=fondo. Deben caer en píxeles válidos.
import matplotlib.pyplot as plt; plt.imshow(img.preview()); plt.show()

In [ ]:
POINTS=[(256,256,1),(100,100,0)]   # EDITAR: al menos un positivo sobre agua visible
out=be.segment(img,POINTS,compare=False)
m=out['frozen']['mask']; print('píxeles de máscara:',int(m.sum()),'estado:',out['status'])
assert m.dtype==bool and m.shape==img.valid.shape and m.sum()>0,'La máscara salió vacía: mover los puntos'
from PIL import Image; Image.fromarray((m*255).astype('uint8')).save('/content/e3_demo_mask.png')
json.dump({'points':POINTS,'mask_pixels':int(m.sum()),'selection':str(out['frozen']['selection']),'gpu':torch.cuda.get_device_name(0),
  'chip_sha256':hashlib.sha256(open('/content/chip.tif','rb').read()).hexdigest(),'checkpoint':'facebook/sam2.1-hiera-tiny (verificado por SHA contra report.json)'},open('/content/e3_demo_smoke.json','w'),indent=1)
print('OK e3_demo_smoke')

**Guardar:** descarga el JSON y conserva la máscara en Drive privado mientras se aclara la licencia del dataset.
